## measles cases around the world by tidyteusday

In [33]:
# import the  libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np      
import os
import warnings
warnings.filterwarnings('ignore')

# Add Plotly for interactive visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# load the  yearly data
yearly_measles = pd.read_csv('../data/cases_year.csv')  
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population
0,AFRO,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02
1,AFRO,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15
2,AFRO,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12
3,AFRO,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11
4,AFRO,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08


| Variable | Class | Description |
|---|---|---|
| region | character | Region name |
| country | character | Country name |
| iso3 | character | Three-letter country code |
| year | character | Year |
| total_population | character | Country population |
| annualized_population_most_recent_year_only | character | Annualized population (2025) |
| total_suspected_measles_rubella_cases | character | Suspected measles/rubella cases: a suspected case is one in which a patient has fever and maculopapular (non-vesicular) rash, or in whom a health-care worker suspects measles (or rubella). |
| measles_total | character | Total measles cases: sum of clinically compatible, epidemiologically linked, and laboratory-confirmed cases. |
| measles_lab_confirmed | character | Laboratory-confirmed measles cases: a suspected measles case confirmed positive by testing in a proficient laboratory, with vaccine-associated illness ruled out. |
| measles_epi_linked | character | Epidemiologically linked measles cases: a suspected measles case not laboratory-confirmed, but geographically and temporally related (rash onset 7–23 days apart) to a laboratory-confirmed or another epidemiologically linked measles case. |
| measles_clinical | character | Clinically compatible measles cases: a suspected case with fever and maculopapular (non-vesicular) rash and at least one of cough, coryza, or conjunctivitis, with no adequate specimen and no epidemiological link to a laboratory-confirmed measles case or other communicable disease. |
| measles_incidence_rate_per_1000000_total_population | character | Measles cases per million population |
| rubella_total | character | Total rubella cases |
| rubella_lab_confirmed | character | Laboratory-confirmed rubella cases |
| rubella_epi_linked | character | Epidemiologically linked rubella cases |
| rubella_clinical | character | Clinically compatible rubella cases |
| rubella_incidence_rate_per_1000000_total_population | character | Rubella cases per million population |
| discarded_cases | character | Discarded cases: a suspected case investigated and discarded as non-measles (and non-rubella). |
| discarded_non_measles_rubella_cases_per_100000_total_population | character | Discarded cases per million population |

In [4]:
yearly_measles.info()

<class 'pandas.DataFrame'>
RangeIndex: 2382 entries, 0 to 2381
Data columns (total 19 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   region                                                           2382 non-null   str    
 1   country                                                          2380 non-null   str    
 2   iso3                                                             2382 non-null   str    
 3   year                                                             2382 non-null   int64  
 4   total_population                                                 2382 non-null   int64  
 5   annualized_population_most_recent_year_only                      2382 non-null   int64  
 6   total_suspected_measles_rubella_cases                            2295 non-null   float64
 7   measles_total                                        

In [5]:
# check for duplicates
yearly_measles.duplicated().sum()

np.int64(0)

## cleaning the data

In [6]:
# standardize the column names 
yearly_measles.columns = yearly_measles.columns.str.strip().str.lower().str.replace(' ', '_')
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population
0,AFRO,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02
1,AFRO,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15
2,AFRO,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12
3,AFRO,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11
4,AFRO,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08


In [7]:
#normalize region codes 
region_map = {
    "AFRO": "AFR", "EURO": "EUR", "AMRO": "AMR",
    "EMRO": "EMR", "SEARO": "SEAR", "WPRO": "WPR"
}
yearly_measles["region"] = yearly_measles["region"].replace(region_map)

In [9]:
#separate menaingfull nulls from the rest
surveillance_cols = [
    "measles_lab_confirmed", "measles_epi_linked", "measles_clinical",
    "rubella_lab_confirmed", "rubella_epi_linked", "rubella_clinical",
    "discarded_cases"
]
yearly_measles["surveillance_reported"] = yearly_measles[surveillance_cols].notna().all(axis=1)
# print head 
yearly_measles.head()

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,measles_clinical,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population,surveillance_reported
0,AFR,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,53,1.46,13,13,0,0,0.35,8.0,0.02,True
1,AFR,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,0,0.00,29,29,0,0,0.75,56.0,0.15,True
2,AFR,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,0,0.00,3,3,0,0,0.08,46.0,0.12,True
3,AFR,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,0,1.55,2,2,0,0,0.05,45.0,0.11,True
4,AFR,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,1,1.20,11,11,0,0,0.27,33.0,0.08,True


In [11]:
# create a new column to identify rows where measles cases are reported but lab confirmation is missing
yearly_measles["measles_breakdown_missing"] = (
    yearly_measles["measles_total"] > 0
) & (yearly_measles["measles_lab_confirmed"].isna())
yearly_measles.head()  

,region,country,iso3,year,total_population,annualized_population_most_recent_year_only,total_suspected_measles_rubella_cases,measles_total,measles_lab_confirmed,measles_epi_linked,...,measles_incidence_rate_per_1000000_total_population,rubella_total,rubella_lab_confirmed,rubella_epi_linked,rubella_clinical,rubella_incidence_rate_per_1000000_total_population,discarded_cases,discarded_non_measles_rubella_cases_per_100000_total_population,surveillance_reported,measles_breakdown_missing
0,AFR,Algeria,DZA,2012,37646166,37646166,76.0,55,2,0,...,1.46,13,13,0,0,0.35,8.0,0.02,True,False
1,AFR,Algeria,DZA,2013,38414172,38414172,85.0,0,0,0,...,0.00,29,29,0,0,0.75,56.0,0.15,True,False
2,AFR,Algeria,DZA,2014,39205031,39205031,49.0,0,0,0,...,0.00,3,3,0,0,0.08,46.0,0.12,True,False
3,AFR,Algeria,DZA,2015,40019529,40019529,109.0,62,2,60,...,1.55,2,2,0,0,0.05,45.0,0.11,True,False
4,AFR,Algeria,DZA,2016,40850721,40850721,93.0,49,21,27,...,1.20,11,11,0,0,0.27,33.0,0.08,True,False


In [12]:
# Validate: suspected >= measles_total + rubella_total
yearly_measles["suspect_sum"] = yearly_measles["measles_total"] + yearly_measles["rubella_total"]
yearly_measles["suspect_mismatch"] = yearly_measles["total_suspected_measles_rubella_cases"] < yearly_measles["suspect_sum"]
 
print("\n── Data quality flags ──")
print(f"Rows missing surveillance breakdown: {(~yearly_measles['surveillance_reported']).sum()}")
print(f"Rows with measles but no breakdown:  {yearly_measles['measles_breakdown_missing'].sum()}")
print(f"Rows where suspect < total cases:    {yearly_measles['suspect_mismatch'].sum()}")
 


── Data quality flags ──
Rows missing surveillance breakdown: 87
Rows with measles but no breakdown:  0
Rows where suspect < total cases:    13


In [13]:
# Convert data types
numeric_cols = yearly_measles.select_dtypes(include=['object']).columns.difference(['region', 'country', 'iso3'])
for col in numeric_cols:
    yearly_measles[col] = pd.to_numeric(yearly_measles[col], errors='coerce')

yearly_measles['year'] = yearly_measles['year'].astype(int)

# Check data types after conversion
print("\n── Data Types After Conversion ──")
print(yearly_measles.dtypes)
print(f"\nMissing values per column:")
print(yearly_measles.isnull().sum())


── Data Types After Conversion ──
region                                                                 str
country                                                                str
iso3                                                                   str
year                                                                 int64
total_population                                                     int64
annualized_population_most_recent_year_only                          int64
total_suspected_measles_rubella_cases                              float64
measles_total                                                        int64
measles_lab_confirmed                                                int64
measles_epi_linked                                                   int64
measles_clinical                                                     int64
measles_incidence_rate_per_1000000_total_population                float64
rubella_total                                                    

In [17]:
# Fill missing values for surveillance columns with 0 (indicating unreported)
yearly_measles[surveillance_cols] = yearly_measles[surveillance_cols].fillna(0)

# For population fields, forward-fill then backward-fill
yearly_measles['total_population'] = yearly_measles.groupby('country')['total_population'].transform(lambda x: x.ffill().bfill())

print("\n── Missing values after filling ──")
print(yearly_measles.isnull().sum())


── Missing values after filling ──
region                                                               0
country                                                              2
iso3                                                                 0
year                                                                 0
total_population                                                     2
annualized_population_most_recent_year_only                          0
total_suspected_measles_rubella_cases                               87
measles_total                                                        0
measles_lab_confirmed                                                0
measles_epi_linked                                                   0
measles_clinical                                                     0
measles_incidence_rate_per_1000000_total_population                  0
rubella_total                                                        0
rubella_lab_confirmed                    

In [15]:
# Remove duplicates
initial_rows = len(yearly_measles)
yearly_measles = yearly_measles.drop_duplicates(subset=['region', 'country', 'iso3', 'year'])
removed_rows = initial_rows - len(yearly_measles)
print(f"\n── Duplicates Removed ──")
print(f"Removed {removed_rows} duplicate rows")

# Rename cleaned indicator columns for clarity
yearly_measles = yearly_measles.drop(columns=['suspect_sum', 'suspect_mismatch'])

print(f"\n── Final Dataset Summary ──")
print(f"Shape: {yearly_measles.shape}")
print(f"Rows: {yearly_measles.shape[0]}, Columns: {yearly_measles.shape[1]}")


── Duplicates Removed ──
Removed 0 duplicate rows

── Final Dataset Summary ──
Shape: (2382, 21)
Rows: 2382, Columns: 21


In [16]:
# Save cleaned data
output_path = '../data/cleaned/cases_year_clean.csv'
yearly_measles.to_csv(output_path, index=False)
print(f"\n✓ Cleaned yearly data saved to: {output_path}")

# Final validation
print(f"\n── Final Validation ──")
cleaned_data = pd.read_csv(output_path)
print(f"Loaded cleaned data shape: {cleaned_data.shape}")
print(f"\nColumn names:")
print(cleaned_data.columns.tolist())


✓ Cleaned yearly data saved to: ../data/cleaned/cases_year_clean.csv

── Final Validation ──
Loaded cleaned data shape: (2382, 21)

Column names:
['region', 'country', 'iso3', 'year', 'total_population', 'annualized_population_most_recent_year_only', 'total_suspected_measles_rubella_cases', 'measles_total', 'measles_lab_confirmed', 'measles_epi_linked', 'measles_clinical', 'measles_incidence_rate_per_1000000_total_population', 'rubella_total', 'rubella_lab_confirmed', 'rubella_epi_linked', 'rubella_clinical', 'rubella_incidence_rate_per_1000000_total_population', 'discarded_cases', 'discarded_non_measles_rubella_cases_per_100000_total_population', 'surveillance_reported', 'measles_breakdown_missing']


## Exploratory Data Analysis (EDA)

In [18]:
# Descriptive statistics
print("═" * 80)
print("DESCRIPTIVE STATISTICS")
print("═" * 80)
print(yearly_measles.describe())
print("\n")
print(yearly_measles[['measles_total', 'rubella_total', 'measles_lab_confirmed']].describe())

════════════════════════════════════════════════════════════════════════════════
DESCRIPTIVE STATISTICS
════════════════════════════════════════════════════════════════════════════════
              year  total_population  \
count  2382.000000      2.380000e+03   
mean   2018.631402      4.272505e+07   
std       3.958222      1.507132e+08   
min    2012.000000      1.757000e+03   
25%    2015.000000      2.878645e+06   
50%    2019.000000      1.009075e+07   
75%    2022.000000      3.172827e+07   
max    2025.000000      1.463866e+09   

       annualized_population_most_recent_year_only  \
count                                 2.382000e+03   
mean                                  4.074782e+07   
std                                   1.456164e+08   
min                                   7.590000e+02   
25%                                   2.797780e+06   
50%                                   9.620074e+06   
75%                                   3.021081e+07   
max                   

In [19]:
# Cases by region
print("\n" + "═" * 80)
print("MEASLES CASES BY REGION")
print("═" * 80)
region_analysis = yearly_measles.groupby('region').agg({
    'measles_total': ['sum', 'mean', 'max'],
    'rubella_total': ['sum', 'mean'],
    'country': 'nunique',
    'year': 'nunique'
}).round(2)
region_analysis.columns = ['_'.join(col).strip() for col in region_analysis.columns.values]
print(region_analysis)



════════════════════════════════════════════════════════════════════════════════
MEASLES CASES BY REGION
════════════════════════════════════════════════════════════════════════════════
        measles_total_sum  measles_total_mean  measles_total_max  \
region                                                             
AFR                901018             1499.20             213291   
AMR                 58733              179.61              20901   
EMR                517334             1808.86              49571   
EUR                548694              809.28              57332   
SEAR               632557             4245.35              82461   
WPR                434938             1275.48              53906   

        rubella_total_sum  rubella_total_mean  country_nunique  year_nunique  
region                                                                        
AFR                 67690              112.63               46            14  
AMR                     5      

In [21]:
# Temporal trends
print("\n" + "═" * 80)
print("MEASLES CASES OVER TIME")
print("═" * 80)
yearly_trend = yearly_measles.groupby('year').agg({
    'measles_total': 'sum',
    'rubella_total': 'sum',
    'measles_lab_confirmed': 'sum',
    'country': 'nunique'
}).round(0)
yearly_trend.columns = ['Total_Measles', 'Total_Rubella', 'Lab_Confirmed', 'Countries_Reporting']
print(yearly_trend)



════════════════════════════════════════════════════════════════════════════════
MEASLES CASES OVER TIME
════════════════════════════════════════════════════════════════════════════════
      Total_Measles  Total_Rubella  Lab_Confirmed  Countries_Reporting
year                                                                  
2012         112548          61382          46919                  149
2013         178526          67433          76271                  147
2014         290888          35207         101439                  154
2015         247474          22418          70653                  176
2016         180015          18679          46164                  179
2017         168190          10712          43913                  182
2018         276157          17045          90139                  182
2019         541401          47766         118203                  187
2020          93840           7501          36899                  175
2021          59619           74

In [23]:
# Correlation analysis
print("\n" + "═" * 80)
print("CORRELATION ANALYSIS")
print("═" * 80)
correlation_cols = ['measles_total', 'measles_lab_confirmed', 'measles_incidence_rate_per_1000000_total_population',
                    'rubella_total', 'rubella_lab_confirmed', 'total_population']
corr_matrix = yearly_measles[correlation_cols].corr()
print(corr_matrix.round(3))



════════════════════════════════════════════════════════════════════════════════
CORRELATION ANALYSIS
════════════════════════════════════════════════════════════════════════════════
                                                    measles_total  \
measles_total                                               1.000   
measles_lab_confirmed                                       0.464   
measles_incidence_rate_per_1000000_total_popula...          0.499   
rubella_total                                               0.155   
rubella_lab_confirmed                                       0.074   
total_population                                            0.386   

                                                    measles_lab_confirmed  \
measles_total                                                       0.464   
measles_lab_confirmed                                               1.000   
measles_incidence_rate_per_1000000_total_popula...                  0.114   
rubella_total           

In [34]:
# Visualization: Cases by region (Plotly)
region_cases = yearly_measles.groupby('region')['measles_total'].sum().sort_values(ascending=False)
fig = px.bar(x=region_cases.index, y=region_cases.values, 
             labels={'x': 'Region', 'y': 'Number of Cases'},
             title='Total Measles Cases by Region (2012-2025)',
             color=region_cases.values,
             color_continuous_scale='Viridis')
fig.update_layout(showlegend=False, height=600, hovermode='x unified')
fig.show()

In [38]:
# Visualization: Correlation heatmap (Plotly)
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont={"size": 10},
    colorbar=dict(title='Correlation')
))
fig.update_layout(title='Correlation Matrix of Key Variables',
                  height=700, width=900)
fig.show()

In [41]:
# Visualization: Lab-confirmed vs clinical cases (Plotly)
measles_breakdown = yearly_measles.groupby('year')[['measles_lab_confirmed', 'measles_epi_linked', 'measles_clinical']].sum().reset_index()
fig = go.Figure()
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_lab_confirmed'],
                     name='Lab Confirmed', marker_color='#1f77b4'))
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_epi_linked'],
                     name='Epidemiologically Linked', marker_color='#ff7f0e'))
fig.add_trace(go.Bar(x=measles_breakdown['year'], y=measles_breakdown['measles_clinical'],
                     name='Clinical', marker_color='#2ca02c'))
fig.update_layout(title='Measles Cases by Classification Type Over Time',
                  xaxis_title='Year', yaxis_title='Number of Cases',
                  barmode='group', height=600, hovermode='x unified')
fig.show()

In [42]:
# Visualization: Distribution of measles incidence rates (Plotly)
fig = make_subplots(rows=1, cols=2,
                     subplot_titles=('Distribution of Measles Incidence Rate', 
                                     'Measles Incidence Rate by Region'))

# Histogram
fig.add_trace(
    go.Histogram(x=yearly_measles['measles_incidence_rate_per_1000000_total_population'],
                 nbinsx=50, name='Histogram', marker_color='steelblue'),
    row=1, col=1
)

# Boxplot by region
for region in yearly_measles['region'].unique():
    region_data = yearly_measles[yearly_measles['region'] == region]['measles_incidence_rate_per_1000000_total_population']
    fig.add_trace(
        go.Box(y=region_data, name=region),
        row=1, col=2
    )

fig.update_xaxes(title_text='Incidence Rate (per 1,000,000)', row=1, col=1)
fig.update_xaxes(title_text='Region', row=1, col=2)
fig.update_yaxes(title_text='Frequency', row=1, col=1)
fig.update_yaxes(title_text='Incidence Rate (per 1,000,000)', row=1, col=2)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [35]:
# Visualization: Measles cases over time (Plotly)
yearly_trend_plot = yearly_measles.groupby('year')[['measles_total', 'rubella_total']].sum().reset_index()
fig = go.Figure()
fig.add_trace(go.Scatter(x=yearly_trend_plot['year'], y=yearly_trend_plot['measles_total'],
                         mode='lines+markers', name='Measles Total',
                         line=dict(color='#1f77b4', width=3), marker=dict(size=8)))
fig.add_trace(go.Scatter(x=yearly_trend_plot['year'], y=yearly_trend_plot['rubella_total'],
                         mode='lines+markers', name='Rubella Total',
                         line=dict(color='#ff7f0e', width=3), marker=dict(size=8)))
fig.update_layout(title='Measles and Rubella Cases Over Time (2012-2025)',
                  xaxis_title='Year', yaxis_title='Number of Cases',
                  height=600, hovermode='x unified')
fig.show()

In [22]:
# Data quality summary
print("\n" + "═" * 80)
print("DATA QUALITY SUMMARY")
print("═" * 80)
print(f"Total records: {len(yearly_measles):,}")
print(f"Date range: {yearly_measles['year'].min():.0f} - {yearly_measles['year'].max():.0f}")
print(f"Number of regions: {yearly_measles['region'].nunique()}")
print(f"Number of countries: {yearly_measles['country'].nunique()}")
print(f"\nRecords with surveillance data: {yearly_measles['surveillance_reported'].sum():,} ({yearly_measles['surveillance_reported'].sum()/len(yearly_measles)*100:.1f}%)")
print(f"Records with measles breakdown missing: {yearly_measles['measles_breakdown_missing'].sum():,}")



════════════════════════════════════════════════════════════════════════════════
DATA QUALITY SUMMARY
════════════════════════════════════════════════════════════════════════════════
Total records: 2,382
Date range: 2012 - 2025
Number of regions: 6
Number of countries: 193

Records with surveillance data: 2,295 (96.3%)
Records with measles breakdown missing: 0


In [20]:
# Top 15 countries by total measles cases
print("\n" + "═" * 80)
print("TOP 15 COUNTRIES BY TOTAL MEASLES CASES (2012-2025)")
print("═" * 80)
top_countries = yearly_measles.groupby('country').agg({
    'measles_total': 'sum',
    'measles_lab_confirmed': 'sum',
    'rubella_total': 'sum',
    'year': 'nunique'
}).sort_values('measles_total', ascending=False).head(15).round(0)
top_countries.columns = ['Total Measles', 'Lab Confirmed', 'Total Rubella', 'Years Reported']
print(top_countries)



════════════════════════════════════════════════════════════════════════════════
TOP 15 COUNTRIES BY TOTAL MEASLES CASES (2012-2025)
════════════════════════════════════════════════════════════════════════════════
                                  Total Measles  Lab Confirmed  Total Rubella  \
country                                                                         
India                                    474525          78576          33057   
Madagascar                               225947           1745           2108   
Nigeria                                  210073          29000           7363   
China                                    172872         162159         125908   
Philippines                              148930          28448           2160   
Yemen                                    147829           9428          10784   
Ukraine                                  136311          21573           1409   
Pakistan                                 132292         

In [39]:
# Countries that drove the 2019 measles spike (vs 2018)
country_year = (
    yearly_measles
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

pivot_cases = country_year.pivot(index='country', columns='year', values='measles_total').fillna(0)
pivot_cases['increase_2019_vs_2018'] = pivot_cases.get(2019, 0) - pivot_cases.get(2018, 0)

spike_drivers = (
    pivot_cases[pivot_cases['increase_2019_vs_2018'] > 0][['increase_2019_vs_2018']]
    .sort_values('increase_2019_vs_2018', ascending=False)
)

global_spike = spike_drivers['increase_2019_vs_2018'].sum()
top_n = 15
top_spike_drivers = spike_drivers.head(top_n).copy()
top_spike_drivers['share_of_global_spike_pct'] = (
    top_spike_drivers['increase_2019_vs_2018'] / global_spike * 100
).round(2)

print('Countries driving the 2019 spike (increase from 2018 to 2019):')
print(top_spike_drivers)
print(f"\nTotal global increase (2019 vs 2018): {int(global_spike):,}")
print(
    f"Top {top_n} countries explain "
    f"{top_spike_drivers['share_of_global_spike_pct'].sum():.2f}% of that increase."
)

# Plotly horizontal bar chart
plot_df = top_spike_drivers.sort_values('increase_2019_vs_2018', ascending=True).reset_index()
fig = px.bar(plot_df, y='country', x='increase_2019_vs_2018',
             orientation='h',
             title='Top Countries Driving the 2019 Measles Spike (vs 2018)',
             labels={'increase_2019_vs_2018': 'Increase in Measles Cases', 'country': 'Country'},
             color='increase_2019_vs_2018',
             color_continuous_scale='Reds')
fig.update_layout(height=700, showlegend=False, hovermode='y unified')
fig.show()

Countries driving the 2019 spike (increase from 2018 to 2019):
year                              increase_2019_vs_2018  \
country                                                   
Madagascar                                     201239.0   
Philippines                                     25579.0   
Nigeria                                         21466.0   
Democratic Republic of the Congo                12975.0   
Kazakhstan                                      12750.0   
Brazil                                          10575.0   
Viet Nam                                         4955.0   
Tunisia                                          4650.0   
Ukraine                                          4114.0   
Myanmar                                          3856.0   
Bangladesh                                       3216.0   
Iraq                                             3130.0   
Angola                                           2689.0   
Algeria                                          229

In [37]:
# Countries that drove the 2023 to 2024 measles increase
country_year_23_24 = (
    yearly_measles
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

pivot_23_24 = country_year_23_24.pivot(index='country', columns='year', values='measles_total').fillna(0)
pivot_23_24['increase_2024_vs_2023'] = pivot_23_24.get(2024, 0) - pivot_23_24.get(2023, 0)

drivers_23_24 = (
    pivot_23_24[pivot_23_24['increase_2024_vs_2023'] > 0][['increase_2024_vs_2023']]
    .sort_values('increase_2024_vs_2023', ascending=False)
)

global_increase_23_24 = drivers_23_24['increase_2024_vs_2023'].sum()
top_n_23_24 = 15
top_drivers_23_24 = drivers_23_24.head(top_n_23_24).copy()
top_drivers_23_24['share_of_increase_pct'] = (
    top_drivers_23_24['increase_2024_vs_2023'] / global_increase_23_24 * 100
).round(2)

print('Countries driving the 2023 to 2024 increase:')
print(top_drivers_23_24)
print(f"\nTotal global increase (2024 vs 2023): {int(global_increase_23_24):,}")
print(
    f"Top {top_n_23_24} countries explain "
    f"{top_drivers_23_24['share_of_increase_pct'].sum():.2f}% of that increase."
)

# Plotly horizontal bar chart
plot_23_24 = top_drivers_23_24.sort_values('increase_2024_vs_2023', ascending=True).reset_index()
fig = px.bar(plot_23_24, y='country', x='increase_2024_vs_2023',
             orientation='h',
             title='Top Countries Driving Measles Increase (2023 to 2024)',
             labels={'increase_2024_vs_2023': 'Increase in Measles Cases', 'country': 'Country'},
             color='increase_2024_vs_2023',
             color_continuous_scale='Oranges')
fig.update_layout(height=700, showlegend=False, hovermode='y unified')
fig.show()

Countries driving the 2023 to 2024 increase:
year                                                increase_2024_vs_2023  \
country                                                                     
Romania                                                           27320.0   
Iraq                                                              22734.0   
Ethiopia                                                          13696.0   
Kazakhstan                                                        13036.0   
Russian Federation                                                 9116.0   
Thailand                                                           8155.0   
Afghanistan                                                        7240.0   
Kyrgyzstan                                                         6945.0   
Pakistan                                                           6748.0   
Burkina Faso                                                       5712.0   
Côte d'Ivoire                  

In [40]:
# Compare 2019 outbreak countries vs 2023 resurgence countries
comparison_df = pd.DataFrame({
    '2019_spike_rank': top_spike_drivers['increase_2019_vs_2018'],
    '2024_spike_rank': top_drivers_23_24['increase_2024_vs_2023']
}).fillna(0)

comparison_df['appears_in_both'] = (comparison_df['2019_spike_rank'] > 0) & (comparison_df['2024_spike_rank'] > 0)
comparison_df = comparison_df.sort_values('appears_in_both', ascending=False)

print("\n" + "═" * 80)
print("COMPARISON: 2019 OUTBREAK vs 2024 RESURGENCE")
print("═" * 80)
print(f"Countries in 2019 top drivers: {len(top_spike_drivers)}")
print(f"Countries in 2024 top drivers: {len(top_drivers_23_24)}")
print(f"Countries appearing in BOTH outbreaks: {comparison_df['appears_in_both'].sum()}")
print("\nCountries driving both outbreaks:")
print(comparison_df[comparison_df['appears_in_both']].sort_values('2019_spike_rank', ascending=False))

# Plotly scatter plot
comparison_reset = comparison_df.reset_index()
comparison_reset['color'] = comparison_reset['appears_in_both'].map({True: 'Both Outbreaks', False: 'Single Outbreak'})

fig = px.scatter(comparison_reset, x='2019_spike_rank', y='2024_spike_rank',
                 color='color',
                 color_discrete_map={'Both Outbreaks': '#d62728', 'Single Outbreak': '#1f77b4'},
                 size='2019_spike_rank',
                 hover_name='country',
                 title='Comparison: 2019 Outbreak vs 2024 Resurgence',
                 labels={'2019_spike_rank': 'Increase in Cases 2019 vs 2018',
                        '2024_spike_rank': 'Increase in Cases 2024 vs 2023'})

# Add text labels for both-outbreak countries
for idx, row in comparison_reset[comparison_reset['appears_in_both']].iterrows():
    fig.add_annotation(x=row['2019_spike_rank'], y=row['2024_spike_rank'],
                      text=row['country'], showarrow=False, font=dict(size=9))

fig.update_layout(height=700, hovermode='closest')
fig.show()


════════════════════════════════════════════════════════════════════════════════
COMPARISON: 2019 OUTBREAK vs 2024 RESURGENCE
════════════════════════════════════════════════════════════════════════════════
Countries in 2019 top drivers: 15
Countries in 2024 top drivers: 15
Countries appearing in BOTH outbreaks: 4

Countries driving both outbreaks:
            2019_spike_rank  2024_spike_rank  appears_in_both
country                                                      
Kazakhstan          12750.0          13036.0             True
Viet Nam             4955.0           2007.0             True
Iraq                 3130.0          22734.0             True
Ethiopia             2259.0          13696.0             True


In [46]:
median_incidence = (
    yearly_measles
    .groupby(["region", "year"])["measles_incidence_rate_per_1000000_total_population"]
    .median()
    .reset_index(name="median_incidence")
)

fig = px.line(
    median_incidence,
    x="year",
    y="median_incidence",
    color="region",
    markers=True,
    title="Median Measles Incidence Rate by Region Over Time",
    labels={
        "year": "Year",
        "median_incidence": "Median Incidence Rate (per 1,000,000)",
        "region": "Region"
    }
)

fig.update_layout(hovermode="x unified", height=600)
fig.show()

In [44]:
# EMR countries driving the 2023 spike (increase from 2022 to 2023)
emr_data = yearly_measles[yearly_measles['region'] == 'EMR'].copy()

emr_country_year = (
    emr_data
    .groupby(['country', 'year'], as_index=False)['measles_total']
    .sum()
)

emr_pivot = emr_country_year.pivot(index='country', columns='year', values='measles_total').fillna(0)
emr_pivot['increase_2023_vs_2022'] = emr_pivot.get(2023, 0) - emr_pivot.get(2022, 0)

emr_spike_drivers = (
    emr_pivot[emr_pivot['increase_2023_vs_2022'] > 0][['increase_2023_vs_2022']]
    .sort_values('increase_2023_vs_2022', ascending=False)
)

emr_global_spike = emr_spike_drivers['increase_2023_vs_2022'].sum()
emr_top_n = 10
emr_top_drivers = emr_spike_drivers.head(emr_top_n).copy()
emr_top_drivers['share_of_emr_spike_pct'] = (
    emr_top_drivers['increase_2023_vs_2022'] / emr_global_spike * 100
).round(2)

print('EMR countries driving the 2023 spike (increase from 2022 to 2023):')
print(emr_top_drivers)
print(f"\nTotal EMR increase (2023 vs 2022): {int(emr_global_spike):,}")
print(
    f"Top {emr_top_n} countries explain "
    f"{emr_top_drivers['share_of_emr_spike_pct'].sum():.2f}% of the EMR increase."
)

# Interactive Plotly bar chart
emr_plot = emr_top_drivers.sort_values('increase_2023_vs_2022', ascending=True).reset_index()
fig = px.bar(
    emr_plot,
    y='country',
    x='increase_2023_vs_2022',
    orientation='h',
    color='increase_2023_vs_2022',
    color_continuous_scale='Tealgrn',
    title='Top EMR Countries Driving the 2023 Measles Spike (vs 2022)',
    labels={
        'increase_2023_vs_2022': 'Increase in Measles Cases',
        'country': 'Country'
    },
    hover_data={'share_of_emr_spike_pct': True}
)
fig.update_layout(height=600, showlegend=False)
fig.show()

EMR countries driving the 2023 spike (increase from 2022 to 2023):
year                        increase_2023_vs_2022  share_of_emr_spike_pct
country                                                                  
Yemen                                     25647.0                   49.63
Iraq                                       9630.0                   18.64
Pakistan                                   9598.0                   18.57
Sudan                                      2400.0                    4.64
Saudi Arabia                               2013.0                    3.90
Syrian Arab Republic                        527.0                    1.02
Iran (Islamic Republic of)                  414.0                    0.80
United Arab Emirates                        408.0                    0.79
Lebanon                                     260.0                    0.50
Egypt                                       242.0                    0.47

Total EMR increase (2023 vs 2022): 51,675
To

In [45]:
# Compare EMR vs other regions for 2022 to 2023 measles increase
region_year = (
    yearly_measles
    .groupby(['region', 'year'], as_index=False)['measles_total']
    .sum()
)

region_pivot = region_year.pivot(index='region', columns='year', values='measles_total').fillna(0)
region_pivot['increase_2023_vs_2022'] = region_pivot.get(2023, 0) - region_pivot.get(2022, 0)
region_compare = region_pivot[['increase_2023_vs_2022']].reset_index().sort_values('increase_2023_vs_2022', ascending=False)

# Label EMR for highlighting
region_compare['group'] = np.where(region_compare['region'] == 'EMR', 'EMR', 'Other Regions')

print('Regional measles increase (2023 vs 2022):')
print(region_compare[['region', 'increase_2023_vs_2022']])

fig = px.bar(
    region_compare,
    x='region',
    y='increase_2023_vs_2022',
    color='group',
    color_discrete_map={'EMR': '#d62728', 'Other Regions': '#1f77b4'},
    title='EMR vs Other Regions: Measles Increase (2023 vs 2022)',
    labels={'increase_2023_vs_2022': 'Increase in Measles Cases', 'region': 'Region'},
    text='increase_2023_vs_2022'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=550, xaxis=dict(type='category'), yaxis_tickformat=',', hovermode='x unified')
fig.show()

Regional measles increase (2023 vs 2022):
year region  increase_2023_vs_2022
3       EUR                60043.0
4      SEAR                41340.0
2       EMR                33390.0
0       AFR                 8521.0
5       WPR                 4351.0
1       AMR                  -96.0
